# 🛡️ Compliance Intelligence Engine (CIE) — Real-Data Production Training
### Domain-specific LLM · Trained on real enforcement decisions · 5 regulatory frameworks

| | |
|---|---|
| **Base model** | `mistralai/Mistral-7B-Instruct-v0.3` |
| **Method** | QLoRA (4-bit NF4) via Unsloth + SFT + DPO |
| **Dataset** | Real enforcement cases: GDPR · HIPAA · NIST CSF · PCI DSS · ISO 27001 |
| **Data sources** | GDPR Enforcement Tracker · HHS OCR · GAO · PCI SSC · ICO/NCSC |
| **Phases** | Phase 1 — SFT · Phase 2 — DPO alignment |
| **Eval** | Accuracy · ROUGE-L · Hallucination rate · Section coverage |
| **Target GPU** | Kaggle T4 (15.6 GB VRAM) |

---
### 📋 Pipeline
1. Environment install (Unsloth — handles ALL dependency issues automatically)
2. HF authentication
3. CIE system prompt
4. Load real enforcement data (from Kaggle Dataset: `cie-real-data`)
5. Dataset validation & stats
6. Load model + QLoRA via Unsloth
7. Phase 1 — Supervised Fine-Tuning (SFT)
8. Phase 2 — DPO Constitutional Alignment
9. Comprehensive evaluation (accuracy, ROUGE, hallucination)
10. Adversarial testing
11. Merge & export
12. Full training report

> **Before running:** Upload the `data/` folder from your local machine as a Kaggle Dataset
> named **`cie-real-data`** (New Dataset -> Upload folder -> name it `cie-real-data`).
> The files needed are `cie_real_sft.jsonl` and `cie_real_dpo.jsonl`.


## ⚙️ Step 1: Environment Install
> Uses **Unsloth** — purpose-built for Kaggle/Colab T4. Resolves all bitsandbytes, triton, tokenizers, Pillow, and torchvision version conflicts automatically. **Run this cell, then Restart Session, then Run All from Step 2.**

In [ ]:
%%capture
# ── Unsloth: the only reliable way to do QLoRA on Kaggle in 2026 ──
# It bundles pre-compiled CUDA binaries that match Kaggle's driver exactly.
# No manual bitsandbytes/triton/tokenizers pinning needed.
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
# trl<0.9.0 keeps DPOTrainer API stable (DPOConfig interface unchanged)
!pip install --no-deps "trl<0.9.0" peft accelerate
!pip install rouge-score evaluate
!rm -rf /kaggle/working/unsloth_compiled_cache
print("✅ Install complete — now do: Run → Restart Session → Run All")


## 🔐 Step 2: Hugging Face Authentication
Add your token as a Kaggle Secret named `HF_TOKEN`:
`Add-ons → Secrets → Add New Secret`

Mistral-7B-Instruct-v0.3 is Apache 2.0 — no gating. Token only needed for push_to_hub.

In [ ]:
import os, torch

assert torch.cuda.is_available(), "❌ No GPU detected. Enable: Settings → Accelerator → GPU T4 x2"

gpu_name = torch.cuda.get_device_name(0).upper()
is_t4    = any(x in gpu_name for x in ["T4", "V100", "P100", "K80"])
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU      : {gpu_name}")
print(f"VRAM     : {vram_gb:.1f} GB")
print(f"Precision: {'fp16 (T4/Turing)' if is_t4 else 'bf16 (Ampere+)'}")

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
    print("HF token: loaded ✅")
except Exception as e:
    print(f"HF token: not found ({e}). Mistral is public — continuing without it.")
    hf_token = None


## 📋 Step 3: CIE System Prompt

In [ ]:
CIE_SYSTEM_PROMPT = """
You are the Compliance Intelligence Engine (CIE), an advanced compliance, governance,
risk, privacy, cybersecurity, audit, and regulatory analysis system.

You operate as a senior compliance consultant, auditor, governance specialist,
cybersecurity assessor, privacy expert, risk analyst, and control mapping architect.
You must never behave as a generic chatbot.

CORE FRAMEWORKS: ISO 27001/27002, ISO 42001, NIST CSF 2.0, NIST 800-53, NIST AI RMF,
CIS Controls v8, SOC 2, PCI DSS v4.0, COBIT, HITRUST, FedRAMP, CMMC, CJIS,
GDPR, UK GDPR, CCPA/CPRA, HIPAA, HITECH, PIPEDA, LGPD, PIPL, PDPL, DPDP, APPI,
DORA, NIS2, ISO 22301, ISO 31000, EU AI Act, OECD AI Principles,
SOX, GLBA, FCA, SAMA CSF, Saudi ECC, Saudi PDPL, UAE PDPL, MAS, APRA.

OPERATING RULES:
1. Always identify the applicable framework, regulation, article, control, or clause first.
2. Never provide unsupported compliance conclusions.
3. Always explain reasoning behind every determination.
4. Classify compliance as exactly one of: COMPLIANT, PARTIALLY COMPLIANT,
   NON-COMPLIANT, INSUFFICIENT EVIDENCE.
5. Never assume evidence exists. If absent, classify as INSUFFICIENT EVIDENCE.
6. Always identify: assumptions made, risk implications, remediation steps, audit impact.
7. Never fabricate control IDs, article numbers, or legal obligations.
8. When uncertain, state uncertainty explicitly.

MANDATORY RESPONSE FORMAT:
## Executive Summary
## Applicable Frameworks & Controls
## Analysis
## Evidence Found
## Evidence Missing
## Risk Assessment
Likelihood: [Low|Medium|High|Critical]
Impact: [Low|Medium|High|Critical]
Risk Score: [1-100]
Risk Category: [Operational|Security|Privacy|Regulatory|Financial|Reputational]
## Compliance Determination
[COMPLIANT|PARTIALLY COMPLIANT|NON-COMPLIANT|INSUFFICIENT EVIDENCE]
## Cross-Framework Mapping
## Recommended Remediation
## Audit Impact
Finding: [Critical|Major|Minor|Observation]
## Confidence Score
[1-100]/100
"""
print(f"System prompt: {len(CIE_SYSTEM_PROMPT):,} chars ✅")


## 📚 Step 4: Load Real Enforcement Data

**What makes this real-data training better than synthetic:**
- Grounded in **actual regulatory enforcement decisions** (not generated scenarios)
- Covers real breach investigations: Target, Anthem, Yahoo, NHS WannaCry, LastPass
- Real GDPR fines with actual DPA reasoning (Meta $1.2B, Google $150M, BA $20M)
- Real HIPAA OCR settlements with documented failure modes
- Real NIST CSF audit findings from GAO and CISA assessments
- DPO pairs teach the model what **not** to say (vague chatbot vs structured analysis)

**Data sources (all public record):**
| Framework | Source | Cases |
|---|---|---|
| GDPR | GDPR Enforcement Tracker + EDPB | 9 |
| HIPAA | HHS OCR Enforcement Actions | 8 |
| NIST CSF | GAO/CISA Audit Reports | 8 |
| PCI DSS | Public Breach Investigations | 7 |
| ISO 27001 | ICO/NCSC Decisions | 7 |

> **Required:** Upload the `data/` folder as a Kaggle Dataset named `cie-real-data`


In [ ]:
import json, os
from datasets import Dataset

# ── Kaggle dataset path ────────────────────────────────────────────────────────
# Add the dataset to this notebook via:
# Notebook Settings -> Add Data -> search "cie-real-data" -> Add
DATA_DIR = "/kaggle/input/cie-real-data"
SFT_FILE = f"{DATA_DIR}/cie_real_sft.jsonl"
DPO_FILE = f"{DATA_DIR}/cie_real_dpo.jsonl"

assert os.path.exists(SFT_FILE), (
    f"SFT file not found: {SFT_FILE}
"
    "Did you add the 'cie-real-data' dataset to this notebook?"
)
assert os.path.exists(DPO_FILE), (
    f"DPO file not found: {DPO_FILE}
"
    "Did you add the 'cie-real-data' dataset to this notebook?"
)

with open(SFT_FILE, encoding="utf-8") as f:
    sft_ds_raw = [json.loads(line) for line in f if line.strip()]
with open(DPO_FILE, encoding="utf-8") as f:
    dpo_ds_raw = [json.loads(line) for line in f if line.strip()]

print(f"SFT records loaded : {len(sft_ds_raw):,}")
print(f"DPO pairs loaded   : {len(dpo_ds_raw):,}")

# Framework distribution
framework_counts = {}
for r in sft_ds_raw:
    fw = r.get("framework", r.get("domain", "Unknown"))
    framework_counts[fw] = framework_counts.get(fw, 0) + 1
print("
Framework distribution:")
for fw, count in sorted(framework_counts.items(), key=lambda x: -x[1]):
    print(f"  {fw:<20} {count:>4}")

# Verdict distribution
verdict_counts = {}
for r in sft_ds_raw:
    v = r.get("verdict", "Unknown")
    verdict_counts[v] = verdict_counts.get(v, 0) + 1
print("
Verdict distribution:", verdict_counts)

# Source breakdown
source_counts = {}
for r in sft_ds_raw:
    s = r.get("source", "unknown")
    source_counts[s] = source_counts.get(s, 0) + 1
real_keys = {"real_enforcement", "real_audit", "real_breach", "real_incident"}
real_total = sum(v for k, v in source_counts.items() if k in real_keys)
print(f"
Real enforcement cases : {real_total}")
print(f"Augmented examples     : {source_counts.get('augmented', 0)}")
print("
Real enforcement data loaded successfully.")



## ✅ Step 5: Dataset Validation & Train/Eval Split

In [ ]:
import numpy as np
from collections import Counter
from datasets import Dataset

lengths = [len(r["text"].split()) for r in sft_ds_raw]
print("=== SFT Dataset ===")
print(f"Total examples : {len(sft_ds_raw):,}")
print(f"Avg length     : {np.mean(lengths):.0f} words")
print(f"Max length     : {np.max(lengths):,} words")
print(f"P95 length     : {np.percentile(lengths, 95):.0f} words")

REQUIRED_SECTIONS = [
    "Executive Summary", "Applicable Frameworks", "Analysis",
    "Evidence Found", "Risk Assessment", "Compliance Determination",
    "Recommended Remediation", "Audit Impact", "Confidence Score"
]
missing = sum(1 for r in sft_ds_raw if any(s not in r["text"] for s in REQUIRED_SECTIONS))
print(f"
Section validation")
print(f"Records with all 9 sections : {len(sft_ds_raw) - missing:,}")
print(f"Records missing sections    : {missing}")
print(f"
DPO pairs: {len(dpo_ds_raw):,}")

sft_hf   = Dataset.from_list([{"text": r["text"]} for r in sft_ds_raw])
split    = sft_hf.train_test_split(test_size=0.1, seed=42)
train_ds = split["train"]
eval_ds  = split["test"]

dpo_hf    = Dataset.from_list(dpo_ds_raw)
dpo_split = dpo_hf.train_test_split(test_size=0.1, seed=42)
dpo_train = dpo_split["train"]
dpo_eval  = dpo_split["test"]

print(f"
SFT train : {len(train_ds):,} | SFT eval : {len(eval_ds):,}")
print(f"DPO train : {len(dpo_train):,} | DPO eval : {len(dpo_eval):,}")
print("
Validation complete. Real enforcement dataset ready for training.")



## 🤖 Step 6: Load Base Model with QLoRA (4-bit NF4) via Unsloth

In [ ]:
# MUST be the FIRST import — Unsloth patches torch before anything else loads
import unsloth
from unsloth import FastLanguageModel
import torch, gc

torch.cuda.empty_cache()
gc.collect()

MODEL_ID      = "mistralai/Mistral-7B-Instruct-v0.3"
MAX_SEQ_LEN   = 2048

print(f"Loading {MODEL_ID} via Unsloth (4-bit QLoRA)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_ID,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit   = True,   # NF4 quantization — Unsloth handles the CUDA binary
    token          = hf_token,
)

# LoRA — all attention + FFN layers
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    lora_alpha     = 32,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout   = 0.05,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",  # Unsloth's memory-efficient checkpointing
    random_state   = 42,
)

tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

total    = sum(p.numel() for p in model.parameters())
trainable= sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters  : {total/1e9:.2f}B total  |  {trainable:,} trainable ({100*trainable/total:.3f}%)")
print(f"VRAM used   : {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"VRAM free   : {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9:.2f} GB")


## 🏋️ Step 7: Phase 1 — Supervised Fine-Tuning (SFT)

In [ ]:
import time
from transformers import TrainingArguments
from trl import SFTTrainer

SFT_OUTPUT_DIR = "/kaggle/working/cie-sft"

sft_args = TrainingArguments(
    output_dir                  = SFT_OUTPUT_DIR,
    num_train_epochs            = 3,
    per_device_train_batch_size = 1,
    per_device_eval_batch_size  = 1,
    gradient_accumulation_steps = 8,
    gradient_checkpointing      = True,
    optim                       = "adamw_8bit",   # Unsloth's built-in 8-bit Adam
    learning_rate               = 2e-4,
    lr_scheduler_type           = "cosine",
    warmup_steps                = 100,
    weight_decay                = 0.01,
    max_grad_norm               = 0.3,
    fp16                        = is_t4,          # T4/V100: True
    bf16                        = (not is_t4),    # A100/A10: True
    logging_steps               = 25,
    eval_strategy               = "steps",        # renamed from evaluation_strategy in newer transformers
    eval_steps                  = 100,
    save_strategy               = "steps",
    save_steps                  = 100,
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    report_to                   = "none",
)

sft_trainer = SFTTrainer(
    model               = model,
    train_dataset       = train_ds,
    eval_dataset        = eval_ds,
    tokenizer           = tokenizer,              # renamed from tokenizer= in trl >= 0.9
    args                = sft_args,
    dataset_text_field  = "text",
    max_seq_length      = MAX_SEQ_LEN,
    packing             = False,
)

print("Phase 1 — SFT Training")
print(f"  Train examples     : {len(train_ds):,}")
print(f"  Eval examples      : {len(eval_ds):,}")
print(f"  Epochs             : {sft_args.num_train_epochs}")
print(f"  Effective batch    : {sft_args.per_device_train_batch_size * sft_args.gradient_accumulation_steps}")
print(f"  Precision          : {'fp16' if is_t4 else 'bf16'}")
print("Starting...")

t0         = time.time()
sft_result = sft_trainer.train()
sft_time   = time.time() - t0

sft_trainer.save_model(SFT_OUTPUT_DIR)
tokenizer.save_pretrained(SFT_OUTPUT_DIR)

print(f"\nPhase 1 complete.")
print(f"  Train loss : {sft_result.training_loss:.4f}")
print(f"  Time       : {sft_time/60:.1f} minutes")
print(f"  Saved to   : {SFT_OUTPUT_DIR}")


## 🎯 Step 8: Phase 2 — DPO Constitutional Alignment

**Why DPO matters for a compliance AI:**
SFT teaches the model *what* to say. DPO teaches it *what not to say* — specifically:
- Never fabricate controls or article numbers
- Never give confident verdicts without evidence
- Never behave like a generic chatbot
- Always respond with structured, evidence-grounded output


In [ ]:
from trl import DPOTrainer, DPOConfig
from peft import PeftModel
from transformers import AutoModelForCausalLM
from unsloth import FastLanguageModel

torch.cuda.empty_cache()
gc.collect()

# Reload base + SFT adapter for DPO
print("Loading SFT model for DPO alignment...")
dpo_model, _ = FastLanguageModel.from_pretrained(
    model_name     = SFT_OUTPUT_DIR,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit   = True,
    token          = hf_token,
)
dpo_model = FastLanguageModel.get_peft_model(
    dpo_model,
    r              = 8,
    lora_alpha     = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout   = 0.05,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
)
dpo_model.config.use_cache = False

DPO_OUTPUT_DIR = "/kaggle/working/cie-dpo"

dpo_config = DPOConfig(
    output_dir                  = DPO_OUTPUT_DIR,
    num_train_epochs            = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 8,
    gradient_checkpointing      = True,
    optim                       = "adamw_8bit",
    learning_rate               = 5e-5,
    lr_scheduler_type           = "cosine",
    warmup_steps                = 20,
    max_grad_norm               = 0.3,
    fp16                        = is_t4,
    bf16                        = (not is_t4),
    beta                        = 0.1,
    max_length                  = 1024,
    max_prompt_length           = 512,
    logging_steps               = 10,
    save_strategy               = "epoch",
    report_to                   = "none",
)

dpo_trainer = DPOTrainer(
    model         = dpo_model,
    ref_model     = None,
    tokenizer     = tokenizer,
    train_dataset = dpo_train,
    eval_dataset  = dpo_eval,
    args          = dpo_config,
)

print("Phase 2 — DPO Constitutional Alignment")
print(f"  DPO pairs    : {len(dpo_train):,} train / {len(dpo_eval):,} eval")
print(f"  Beta         : {dpo_config.beta}")
print(f"  Learning rate: {dpo_config.learning_rate}")
print("Starting...")

t0         = time.time()
dpo_result = dpo_trainer.train()
dpo_time   = time.time() - t0
dpo_trainer.save_model(DPO_OUTPUT_DIR)

print(f"\nPhase 2 complete.")
print(f"  DPO loss : {dpo_result.training_loss:.4f}")
print(f"  Time     : {dpo_time/60:.1f} minutes")
print(f"  Saved to : {DPO_OUTPUT_DIR}")


## 📊 Step 9: Comprehensive Evaluation

Four metrics that actually matter for a compliance AI:
1. **Verdict accuracy** — does it classify COMPLIANT/NON-COMPLIANT/PARTIAL/INSUFFICIENT correctly?
2. **ROUGE-L** — how structurally similar is the output to reference responses?
3. **Hallucination rate** — does it invent framework names or control IDs that don't exist?
4. **Section coverage** — does every response contain all 9 mandatory CIE sections?


In [ ]:
from rouge_score import rouge_scorer as rs_module
import re

torch.cuda.empty_cache()
gc.collect()

# Load final DPO model for inference
print("Loading final DPO model for evaluation...")
eval_model, eval_tokenizer = FastLanguageModel.from_pretrained(
    model_name     = DPO_OUTPUT_DIR,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit   = True,
    token          = hf_token,
)
FastLanguageModel.for_inference(eval_model)

def cie_inference(text, max_new_tokens=512):
    prompt = (
        f"<s>[INST] <<SYS>>\n{CIE_SYSTEM_PROMPT}\n<</SYS>>\n\n"
        f"Assess the following practice for compliance:\n\n{text} [/INST] "
    )
    inputs = eval_tokenizer(prompt, return_tensors="pt").to(eval_model.device)
    with torch.no_grad():
        out = eval_model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=0.1, top_p=0.9, do_sample=True,
            pad_token_id=eval_tokenizer.eos_token_id, repetition_penalty=1.1
        )
    return eval_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

EVAL_CASES = [
    ("No penetration test has been conducted in 3 years. PCI DSS environment.",            "NON-COMPLIANT",        "Critical"),
    ("RBAC enforced, MFA mandatory, quarterly access reviews, same-day deprovisioning.",   "COMPLIANT",            "Observation"),
    ("Personal data transferred to US vendor. No SCCs or adequacy decision in place.",     "NON-COMPLIANT",        "Critical"),
    ("Annual DR test. RTO 2hrs. No offsite backup validation.",                            "PARTIALLY COMPLIANT",  "Minor"),
    ("We are compliant.",                                                                   "INSUFFICIENT EVIDENCE","Observation"),
    ("AES-256 at rest, TLS 1.3 in transit, documented key rotation every 12 months.",     "COMPLIANT",            "Observation"),
    ("Production deployments occur without CAB approval. No rollback plan.",               "NON-COMPLIANT",        "Critical"),
    ("Security training completed at onboarding. No annual refresher. 70% staff trained.","PARTIALLY COMPLIANT",  "Minor"),
]

scorer         = rs_module.RougeScorer(["rougeL"], use_stemmer=True)
VALID_CONTROLS = ["ISO 27001", "NIST", "SOC 2", "CIS Control", "GDPR", "PCI DSS",
                  "HIPAA", "DORA", "NIS2", "EU AI Act", "ISO 42001", "SAMA", "CCPA"]
REQUIRED_SECTIONS = [
    "Executive Summary", "Applicable Frameworks", "Analysis",
    "Evidence Found", "Risk Assessment", "Compliance Determination",
    "Recommended Remediation", "Audit Impact", "Confidence Score"
]

results = []
print(f"Running evaluation on {len(EVAL_CASES)} held-out cases...\n")

for i, (scenario, expected_verdict, expected_finding) in enumerate(EVAL_CASES):
    output = cie_inference(scenario)
    verdicts = ["NON-COMPLIANT","PARTIALLY COMPLIANT","COMPLIANT","INSUFFICIENT EVIDENCE"]
    detected = next((v for v in verdicts if v in output.upper()), "UNKNOWN")
    verdict_correct = (detected == expected_verdict)

    ref_response = build_response(
        list(DOMAINS.keys())[i % len(DOMAINS)],
        scenario, expected_verdict, "High", "High", 70, "Security",
        ["Remediate immediately."], expected_finding
    )
    rouge_l = scorer.score(ref_response, output)["rougeL"].fmeasure

    cited_frameworks = re.findall(r"(ISO \d+|NIST [A-Z0-9-]+|SOC 2|CIS Control|GDPR|PCI DSS|HIPAA|DORA|NIS2)", output)
    hallucinated     = [f for f in cited_frameworks if not any(v in f for v in VALID_CONTROLS)]
    hallucination_rate = len(hallucinated) / max(len(cited_frameworks), 1)
    sections_present   = sum(1 for s in REQUIRED_SECTIONS if s in output)
    section_score      = sections_present / len(REQUIRED_SECTIONS)

    results.append({
        "scenario": scenario[:60], "expected": expected_verdict, "detected": detected,
        "verdict_correct": verdict_correct, "rouge_l": rouge_l,
        "hallucination_rate": hallucination_rate, "section_score": section_score,
        "sections_present": sections_present,
    })
    status = "✅" if verdict_correct else "❌"
    print(f"[{i+1}] {status} Expected: {expected_verdict:<25} Got: {detected}")
    print(f"     ROUGE-L: {rouge_l:.3f}  |  Hallucinations: {len(hallucinated)}  |  Sections: {sections_present}/{len(REQUIRED_SECTIONS)}")
    print()

accuracy          = sum(r["verdict_correct"] for r in results) / len(results)
avg_rouge         = sum(r["rouge_l"] for r in results) / len(results)
avg_hallucination = sum(r["hallucination_rate"] for r in results) / len(results)
avg_section       = sum(r["section_score"] for r in results) / len(results)

print("=" * 55)
print("EVALUATION RESULTS")
print("=" * 55)
print(f"Verdict Accuracy    : {accuracy*100:.1f}%")
print(f"Avg ROUGE-L         : {avg_rouge:.3f}")
print(f"Hallucination Rate  : {avg_hallucination*100:.1f}%  (lower is better)")
print(f"Section Coverage    : {avg_section*100:.1f}%")
print()

THRESHOLDS = {
    "Verdict Accuracy": (accuracy,           0.80, "≥80% to pass"),
    "ROUGE-L":          (avg_rouge,           0.30, "≥0.30 to pass"),
    "Hallucination":    (1-avg_hallucination, 0.95, "≤5% hallucination to pass"),
    "Section Coverage": (avg_section,         0.90, "≥90% to pass"),
}
all_pass = True
for metric, (value, threshold, desc) in THRESHOLDS.items():
    passed   = value >= threshold
    all_pass = all_pass and passed
    print(f"  {"✅" if passed else "❌"} {metric:<22} {value:.3f}  ({desc})")

print()
print("OVERALL:", "✅ PRODUCTION READY" if all_pass else "❌ NEEDS MORE TRAINING")


## 🔴 Step 10: Adversarial Testing

Testing the model against inputs specifically designed to expose failure modes:
- **Fabrication traps** — asking about made-up framework names
- **Confidence traps** — vague inputs that should trigger INSUFFICIENT EVIDENCE
- **Contradiction traps** — conflicting evidence in the same scenario
- **Scope traps** — mixing frameworks that don't apply


In [ ]:
ADVERSARIAL_CASES = [
    {
        "name": "Fabrication trap — fake framework",
        "input": "We are certified under ISO 99999 and XYZ-SEC-2024. Are we compliant?",
        "expect": "should NOT cite ISO 99999 or XYZ-SEC as real frameworks",
        "fail_if": ["ISO 99999", "XYZ-SEC-2024"],
    },
    {
        "name": "Confidence trap — vague assertion",
        "input": "We have good security.",
        "expect": "INSUFFICIENT EVIDENCE",
        "fail_if": ["COMPLIANT", "NON-COMPLIANT"],
    },
    {
        "name": "Contradiction trap — mixed evidence",
        "input": "MFA is enforced on all systems. However, the CEO and CFO are exempt from MFA for convenience.",
        "expect": "PARTIALLY COMPLIANT or NON-COMPLIANT — exemptions invalidate the control",
        "fail_if": ["## Compliance Determination\nCOMPLIANT"],
    },
    {
        "name": "Chatbot regression — generic question",
        "input": "What is GDPR?",
        "expect": "structured response, not a chatbot definition",
        "fail_if": ["GDPR stands for", "General Data Protection Regulation is a"],
    },
    {
        "name": "Scope trap — inapplicable framework",
        "input": "Our mobile game app processes no financial data. Are we PCI DSS compliant?",
        "expect": "should note PCI DSS is not applicable or out of scope",
        "fail_if": ["NON-COMPLIANT\n##"],
    },
]

print("Adversarial Evaluation\n" + "="*50)
adv_pass = 0
for case in ADVERSARIAL_CASES:
    output = cie_inference(case["input"], max_new_tokens=400)
    failed = [f for f in case["fail_if"] if f.upper() in output.upper()]
    passed = len(failed) == 0
    adv_pass += int(passed)
    print(f"\n[{"✅ PASS" if passed else "❌ FAIL"}] {case["name"]}")
    print(f"  Expected : {case["expect"]}")
    if failed:
        print(f"  Triggered: {failed}")
    print(f"  Output   : {output[:200]}...")

print(f"\nAdversarial score: {adv_pass}/{len(ADVERSARIAL_CASES)}")
print("✅ Model is robust" if adv_pass >= 4 else "⚠️  Model needs DPO refinement")


## 💾 Step 11: Merge LoRA Weights & Export

In [ ]:
MERGED_DIR = "/kaggle/working/cie-production-final"

torch.cuda.empty_cache()
gc.collect()

print("Merging LoRA adapter weights into base model...")
# Unsloth's merge is faster and produces clean safetensors
merged_model, merged_tokenizer = FastLanguageModel.from_pretrained(
    model_name     = DPO_OUTPUT_DIR,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit   = False,          # load in full precision for merging
    token          = hf_token,
)
merged_model.save_pretrained_merged(MERGED_DIR, merged_tokenizer, save_method="merged_16bit")

print(f"Merged model saved to: {MERGED_DIR}")
print()
print("Deployment options:")
print("  1. HF Hub  : push_to_hub(REPO_NAME) — uncomment cell below")
print("  2. Ollama  : save_pretrained_gguf() then: ollama create cie -f Modelfile")
print("  3. vLLM    : vllm serve cie-production-final")
print("  4. Next    : add RAG layer with pinecone/chroma indexing regulatory text")


In [ ]:
# ── OPTIONAL: Push to Hugging Face Hub ────────────────────────────────────────
# REPO_NAME = "your-username/cie-compliance-7b-production"
# merged_model.push_to_hub(REPO_NAME, token=hf_token, private=True)
# merged_tokenizer.push_to_hub(REPO_NAME, token=hf_token, private=True)
# print(f"Published: https://huggingface.co/{REPO_NAME}")


## 📋 Step 12: Full Production Training Report

In [ ]:
import json

total_time = sft_time + dpo_time

report = {
    "model": {
        "base"           : MODEL_ID,
        "method"         : "QLoRA (4-bit NF4) via Unsloth + SFT + DPO",
        "lora_rank"      : 16,
        "lora_alpha"     : 32,
        "target_modules" : "q/k/v/o/gate/up/down proj (all attention + FFN)",
    },
    "dataset": {
        "sft_examples"   : len(sft_ds_raw),
        "dpo_pairs"      : len(dpo_ds_raw),
        "domains"        : list(DOMAINS.keys()) + ["Constitutional"],
        "domain_count"   : len(DOMAINS) + 1,
    },
    "training": {
        "sft_epochs"     : 3,
        "dpo_epochs"     : 1,
        "sft_loss"       : round(sft_result.training_loss, 4),
        "dpo_loss"       : round(dpo_result.training_loss, 4),
        "total_time_mins": round(total_time / 60, 1),
        "precision"      : "fp16" if is_t4 else "bf16",
        "optimizer"      : "adamw_8bit",
    },
    "evaluation": {
        "verdict_accuracy"  : f"{accuracy*100:.1f}%",
        "rouge_l"           : f"{avg_rouge:.3f}",
        "hallucination_rate": f"{avg_hallucination*100:.1f}%",
        "section_coverage"  : f"{avg_section*100:.1f}%",
        "adversarial_score" : f"{adv_pass}/{len(ADVERSARIAL_CASES)}",
        "production_ready"  : all_pass,
    },
    "output": {
        "sft_model"   : SFT_OUTPUT_DIR,
        "dpo_model"   : DPO_OUTPUT_DIR,
        "merged_model": MERGED_DIR,
    }
}

print(json.dumps(report, indent=2))

with open("/kaggle/working/cie_training_report.json", "w") as f:
    json.dump(report, f, indent=2)
print("\nReport saved: /kaggle/working/cie_training_report.json")
print(f"\nStatus: {"✅ PRODUCTION READY" if all_pass else "❌ REVIEW METRICS ABOVE"}")


---
## 🗺️ What's Next (Post-Kaggle)

| Phase | What | Why |
|-------|------|-----|
| **RAG** | Index full regulatory text (GDPR, ISO 27001, NIST 800-53) into a vector DB | Grounds answers in actual source material — eliminates residual hallucination |
| **Serving** | Deploy with vLLM + FastAPI | Production-grade throughput and latency |
| **UI** | Build a compliance assessment portal on top of the API | Makes the model accessible to non-technical auditors |
| **Monitoring** | Log all queries + verdicts, track drift | Catch model degradation in production |
| **v2 training** | Fine-tune on real audit findings + your own case data | The model gets smarter from every real assessment it sees |

*CIE Production Model — 2-phase training (SFT + DPO) · 12 domains · 2,000+ examples · Evaluated on accuracy, ROUGE, hallucination, adversarial robustness*
